In [7]:
import osmnx as ox
import pandas as pd
import ast

# Toạ độ của trung tâm thành phố Hồ Chí Minh {north, south, east, west} = {10.79326, 10.76813, 106.70840, 106.67175}
osm_file='data/original/hcm_map.osm'
train_file='data/original/oritrain.csv'

# Xử lý file .osm
G = ox.graph_from_xml(osm_file, simplify=True)
nodes, edges = ox.graph_to_gdfs(G)

# Xử lý nodes
nodes = nodes.reset_index()
nodes_df = nodes[['osmid', 'y', 'x']].copy()
nodes_df['osmid'] = nodes_df['osmid'].astype('int64')
nodes_df['y'] = nodes_df['y'].astype('float64')
nodes_df['x'] = nodes_df['x'].astype('float64')
nodes_df.to_csv('data/nodes.csv', index=False)

# Xử lý edges
edges = edges.reset_index()
edges_df = edges[['u', 'v', 'osmid', 'length']].copy()

def extract_first_id(val):
    val_str = str(val)
    if val_str.startswith('['):
        try:
            return int(ast.literal_eval(val_str)[0])
        except:
            pass
    try:
        return int(float(val))
    except:
        return 0
        
edges_df['osmid'] = edges_df['osmid'].apply(extract_first_id)
edges_df['u'] = edges_df['u'].astype('int64')
edges_df['v'] = edges_df['v'].astype('int64')
edges_df['osmid'] = edges_df['osmid'].astype('int64')
edges_df['length'] = edges_df['length'].astype('float64')
edges_df.to_csv('data/edges_raw.csv', index=False)

# Xử lý file train.csv
train_df = pd.read_csv(train_file)
initial_rows = len(train_df)
train_df = train_df[['period', 'LOS', 'street_id']].copy()
train_df['period'] = train_df['period'].astype(str)
train_df['LOS'] = train_df['LOS'].astype(str)
train_df['street_id'] = train_df['street_id'].astype('int64')
los_map = {'A': 1.0, 'B': 1.24, 'C': 1.48, 'D': 1.72, 'E': 1.96, 'F': 2.2}
train_df['los_float'] = train_df['LOS'].map(los_map).fillna(1.0)
train_df.drop(columns=['LOS'], inplace=True)

unique_pairs_count = len(train_df[['street_id', 'period']].drop_duplicates()) 
duplicate_count = initial_rows - unique_pairs_count

train_df = train_df.groupby(['street_id', 'period'], as_index=False)['los_float'].mean()
train_df['los_float'] = train_df['los_float'].round(2)
def get_period_idx(p):
    parts = str(p).replace('period_', '').split('_')
    return int(parts[0]) * 2 + (1 if int(parts[1]) >= 30 else 0)
    
train_df['period_idx'] = train_df['period'].apply(get_period_idx)
train_df.to_csv('data/train.csv', index=False)


In [9]:

train_pivot = train_df.pivot(index='street_id', columns='period_idx', values='los_float')

unique_osmids = set(edges_df['osmid'].unique())
unique_street_ids = set(train_pivot.index)
matched_ids = unique_osmids.intersection(unique_street_ids)

# Tạo danh sách LOS và Weight (48 phần tử)
def build_los_list(osmid):
        if osmid in train_pivot.index:
            row = train_pivot.loc[osmid]
            return [round(row.get(i, 1.0) if pd.notna(row.get(i)) else 1.0, 3) for i in range(48)]
        else:
            return [1.0] * 48

edges_df['los'] = edges_df['osmid'].apply(build_los_list)
edges_df['weight'] = edges_df.apply(lambda r: [round(l * r['length'], 3) for l in r['los']], axis=1)

edges_df.to_csv('data/edges.csv', index=False)
